# octlm on Colab

Runs Day 4 on a Colab GPU. Pick a GPU runtime first, then run the cells in order.

The notebook uses Colab's preinstalled PyTorch, because `uv.lock` pins the CPU build. Colab timings
are not comparable with the CPU timings in `notes/`.

In [ ]:
!nvidia-smi
import torch

print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())

## Repository

For a private repo, put a token in the URL: `https://{TOKEN}@github.com/whynotramaa/octlm.git`.

In [ ]:
import os
import pathlib

REPO = "https://github.com/whynotramaa/octlm.git"
BRANCH = "main"
if pathlib.Path("/content/octlm/.git").exists():
    !cd /content/octlm && git fetch origin && git checkout $BRANCH && git pull
else:
    !git clone --branch $BRANCH $REPO /content/octlm
os.chdir("/content/octlm")
!git log --oneline -1

## Drive

Colab deletes the VM between sessions. Drive keeps the token files, the tokenizer, the checkpoint,
and every JSONL record. The token files are copied to local disk each session, because training
reads them at random offsets and random reads through the Drive mount are slow.

The training stage resumes from the Drive checkpoint when it exists, so rerunning the train cell
after a disconnect continues the run.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/octlm/day4"
!mkdir -p $DRIVE data/tinystories artifacts/day4 runs logs
!cp -n $DRIVE/train.bin $DRIVE/valid.bin data/tinystories/ 2>/dev/null
!cp -n $DRIVE/bpe.json artifacts/day4/ 2>/dev/null
!ls -la data/tinystories artifacts/day4 $DRIVE

In [ ]:
import subprocess


def run(name, command):
    log = open(f"logs/{name}.log", "w")
    stage = ["python", "-m", *command.split()]
    return subprocess.Popen(stage, stdout=log, stderr=subprocess.STDOUT)


def tail(name, lines=15):
    print("".join(open(f"logs/{name}.log").readlines()[-lines:]))

## EXP-065: corpus and tokenizer

Downloads TinyStories V2 (2.2 GB), trains the 8,192-entry BPE on the first 10M characters, measures
encode throughput with and without the chunk cache, and writes `train.bin` and `valid.bin`. Run it
once. Afterwards the Drive cell restores its output.

Watch `logs/prepare.log`. The `encode_benchmark` record comes before the long train-split encode, so
it tells you early how long the encode will take.

In [ ]:
prepare = run("prepare", "octlm.day4 prepare")

In [ ]:
prepare.wait()
tail("prepare")

In [ ]:
!cp data/tinystories/train.bin data/tinystories/valid.bin data/tinystories/prepare.jsonl $DRIVE/
!cp artifacts/day4/bpe.json artifacts/day4/bpe.jsonl $DRIVE/

## EXP-066: mixed precision

Two 1,000-step runs, one mixed and one float32. The mixed run uses bfloat16 on an L4 or A100 and
float16 with `GradScaler` on a T4. Compare validation loss at step 1,000 and `train_seconds`. Each
run has its own checkpoint, so neither one resumes the main run.

In [ ]:
mixed = run(
    "mixed",
    f"octlm.day4 train --stop-after 1000 --checkpoint /content/mixed.pt"
    f" --metrics {DRIVE}/day4-precision.jsonl --samples-out /content/mixed-samples.jsonl",
)
mixed.wait()
full = run(
    "full",
    f"octlm.day4 train --stop-after 1000 --full-precision --checkpoint /content/full.pt"
    f" --metrics {DRIVE}/day4-precision.jsonl --samples-out /content/full-samples.jsonl",
)
full.wait()
!cat $DRIVE/day4-precision.jsonl

## EXP-067: the main run

24,000 steps of 32 x 512 tokens, about 393M tokens. The trainer writes a checkpoint to Drive at
every eval, which is every 1,000 steps. Rerun this cell after a disconnect to resume.

In [ ]:
main = run(
    "main",
    f"octlm.day4 train --checkpoint {DRIVE}/run.pt --metrics {DRIVE}/day4.jsonl"
    f" --samples-out {DRIVE}/day4-samples.jsonl",
)

In [ ]:
tail("main")

## EXP-068: samples

Ten fixed prompts, once greedy and once at temperature 0.8 with top-k 40. Works on any checkpoint,
including one from a run still in progress.

In [ ]:
!python -m octlm.day4 samples --checkpoint $DRIVE/run.pt --samples-out $DRIVE/day4-samples.jsonl --temperature 0
!python -m octlm.day4 samples --checkpoint $DRIVE/run.pt --samples-out $DRIVE/day4-samples.jsonl